# Importing Libraries

In [10]:
import os
import numpy as np
import pandas as pd
import requests
import time

#import matplotlib.pyplot as plt
#import seaborn as sns

# Importing Data - Comtrade API Chili Trade Data

In [7]:

# --- COMTRADE API CONFIGURATION ---
# Reporter Codes: DE (276), UK (826), PL (616); Years: 2020-2025
# HS Codes: Fresh (070960), Dried whole (090421), Dried crushed (090422), Condiments (210390), Preserves (200190)

REPORTERS = "276,826,616"
HS_CODES = "070960,090421,090422,210390,200190"
YEARS = "2020,2021,2022,2023,2024,2025"

def fetch_comtrade_data(reporters, hs_codes, years):
    url = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
    params = {"reporterCode": reporters, "period": years, "cmdCode": hs_codes, "flowCode": "M", "format": "JSON"}
    response = requests.get(url, params=params)
    return pd.DataFrame(response.json()['data']) if response.status_code == 200 else pd.DataFrame()


In [3]:
# -- Functions for Eurostat API dropped - not included in analysis! -- 

# Cleaning Data

In [12]:
# --- CLEANING & RESTRUCTURING ---
df_raw = fetch_comtrade_data(REPORTERS, HS_CODES, YEARS)

# Clean up and drop columns 
cols = ['period', 'reporterDesc', 'partnerDesc', 'cmdCode', 'cmdDesc', 'netWgt', 'primaryValue']
df_clean = df_raw[[c for c in cols if c in df_raw.columns]].copy()
df_clean.columns = ['Year', 'Country', 'Origin', 'HS_Code', 'Description', 'Weight_kg', 'Value_USD']

# Remove "World" agg. data to avoid duplicates 
df_clean = df_clean[df_clean['Origin'] != 'World']

# Export for Diaspora Analysis
df_clean.to_csv("Comstat_Chili_Imports_2020_2025.csv", index=False)
print("Data Collection Complete -- csv created.")

# Find path for my newly created file 
filename = 'Comstat_Chili_Imports_2020_2025.csv' 
full_path = os.path.abspath(filename)

print(f"Chili Import File: {full_path}")

Data Collection Complete -- csv created.
Chili Import File: c:\Users\Kornbichler\Desktop\Comstat_Chili_Imports_2020_2025.csv


## Comtrade data by countries (Fresh capsisum) 
--> see Main repo Github bloodnickel/nairoby-project:main

# Infos on ES & CT API USE and the DS-045409 dataset 
--> Endpoints, Output limits etc. - google

The DS-045409 dataset (EU trade since 1988 by HS2-4-6 and CN8) is a detailed trade database accessible via Eurostat's Comext SDMX REST API and related tools. 
It provides monthly and annual data by individual EU reporter, partner country, and product code. 

Accessing DS-045409 Trade Data

Eurostat Comext API: Data can be accessed via the SDMX 2.1 REST API, which enables automated downloads.
        Base URL: https://ec.europa.eu/eurostat/api/comext/dissemination/sdmx/2.1/dataflow/ESTAT/all
    R Programming (restatapi or comRex): The restatapi package can be used for automation. Additionally, the comRex R package provides a wrapper specifically to access Comext data with ds_id="045409".
    EasyComext (Manual/Batch): DS-045409 is part of the EasyComext user interface, which allows for setting up to 6 parameters for data extraction, including reporter, partner, product, and flow.
    Bulk Download: Eurostat provides a bulk download facility that offers the data in CSV format, with datasets updated regularly to the latest reference month. 

Key Data Features

    Coverage: Detailed data for EU member states, including CN8 (Combined Nomenclature 8-digit) codes.
    Time Period: Monthly and annual trade statistics from 1988 onwards.
    Timezone: It is recommended to define timezones (e.g., UTC, CET) when accessing data through the API to ensure accurate timing of data updates. 

Technical Considerations

    Parameter Updating: The dataset ID can be indicated in query files using the <ds-extid> tag, particularly when transitioning from older datasets.
    Location Changes: The URL structure for Eurostat datasets changes frequently; monitoring the API documentation is advised. 



# Infos on CIF (Import totals) and FOB (Export) values

In the world of trade data, CIF and FOB are "Incoterms" (International Commercial Terms) that define exactly what is included in the dollar value you are seeing in your database.

When you look at trade data for Fresh Capsicum (Bell Peppers), these values tell you whether the "price tag" includes the cost of getting the peppers across the ocean or just the cost of the peppers themselves at the loading dock.

1. CIF Value (Cost, Insurance, and Freight)
This is almost always the value used for Import totals.
    What it represents: The value of the goods plus everything required to get them to the border of the importing country.
    The Formula: CIF=Value of Peppers+Shipping/Freight+Insurance
    Why it matters: If you are analyzing "Import Revenue" or "Import Value," CIF is the "landed" cost. It’s what the importing country actually paid to have that product arrive at their port.

2. FOB Value (Free On Board)
This is the standard value used for Export totals.
    What it represents: The value of the goods at the moment they were loaded onto the ship/truck in the country of origin. It excludes international shipping and insurance.
    The Formula: FOB=Value of Peppers+Cost to load them
    Why it matters: FOB reflects the "true" domestic value of the product for the exporter. It’s the revenue that actually stays in the exporting country’s economy before shipping companies (who might be from a third country) take their cut.

3. The "Capsicum" Context (HS Code 0709.60)
For Fresh Bell Peppers, the distinction is vital because they are perishable.

    HS Code: The base code for fresh/chilled fruits of the genus Capsicum is 0709.60.
    Specifics: You will likely see 0709.60.10 for "Sweet Peppers" (Bell Peppers).

    Data Analysis Gap: If you compare an export record from Mexico (FOB) to an import record in the USA (CIF), the numbers will never match. The USA value will be higher because it includes the cost of the refrigerated trucks and insurance needed to keep the peppers fresh during transit.
    Pro-Tip for your Data Wrangling: If you see a column called cifvalue and another called quantity_kg, dividing them gives you the Unit Price including shipping. If you want to know the "farm-gate" price, you would need the FOB value.



## VARIETIES LEXICON - DATA COLLECTION 
### # CATEGORICAL data to map Varietites to diaspora cuisines 

In [ ]:
# Species and Variety Codifying: Key Capsicum Species & Varieties

SPECIES = ["C. annuum", "C. baccatum", "C. chinense", "C. frutescens", "C. pubescens"] 

# Key Capsicum Species

-C. annuum is by far the largest group, including the sweet Paprika / bell peppers, but also jalapenos and cayennes
-C. chinense - not from China, these include the hottest peppers in the world, like Naga and Reapers
-Additional codifying of Pimientum species --> Todos 

Capsicum annuum: Includes most common varieties like Bell Peppers, Jalapeños, Serrano, Cayenne, and Poblano.
Capsicum chinense: Known for intense heat and fruity flavors, including Habanero, Bhut Jolokia (Ghost Pepper), and Carolina Reaper.
Capsicum baccatum: Primarily South American varieties known for fruity, citrus-like flavors, such as Aji Amarillo and Lemon Drop.
Capsicum frutescens: Famous for small, very hot, upright-growing peppers like Tabasco and Piri Piri.
Capsicum pubescens: Recognizable by their hairy leaves and black seeds, with Rocoto (apple pepper) being a primary example. 

In [15]:
# VARIETIES LEXICON

pepper_data = [
    {"VARIETY": "Bell pepper", "SPECIES": "C. annuum", "SHU": "0", "HS_Fresh": "0709.60.10", "HS_Dry": "0904.21.60", "HS_Ground": "0904.22.20", "Cuisine": "Global", "Trivia": "No heat; high vitamin C"},
    {"VARIETY": "Paprika (Sweet)", "SPECIES": "C. annuum", "SHU": "0-2,500", "HS_Fresh": "0709.60.10", "HS_Dry": "0904.21.20", "HS_Ground": "0904.22.20", "Cuisine": "Hungarian", "Trivia": "Specific cultivar: 'Aji Paprika'"},
    {"VARIETY": "Cayenne", "SPECIES": "C. annuum", "SHU": "30,000-50,000", "HS_Fresh": "0709.60.99", "HS_Dry": "0904.21.90", "HS_Ground": "0904.22.00", "Cuisine": "Global", "Trivia": "Standard for 'red pepper flakes'"},
    {"VARIETY": "Jalapeño", "SPECIES": "C. annuum", "SHU": "2,500-8,000", "HS_Fresh": "0709.60.91", "HS_Dry": "0904.21.60", "HS_Ground": "N/A", "Cuisine": "Mexico", "Trivia": "Often pickled or smoked (Chipotle)"},
    {"VARIETY": "Poblano/Ancho", "SPECIES": "C. annuum", "SHU": "1,000-2,000", "HS_Fresh": "0709.60.99", "HS_Dry": "0904.21.40", "HS_Ground": "0904.22.40", "Cuisine": "Mexico", "Trivia": "Heart-shaped; used in Chiles Rellenos"},
    {"VARIETY": "Aleppo / Pul Biber", "SPECIES": "C. annuum", "SHU": "10,000", "HS_Fresh": "0709.60.99", "HS_Dry": "0904.21.90", "HS_Ground": "0904.22.00", "Cuisine": "Turkish", "Trivia": "De-seeded and salt-cured before grinding"},
    {"VARIETY": "Espelette", "SPECIES": "C. annuum", "SHU": "1,000-4,000", "HS_Fresh": "0709.60.99", "HS_Dry": "0904.21.90", "HS_Ground": "0904.22.00", "Cuisine": "Basque/French", "Trivia": "AOP protected; traditionally hung on houses"},
    {"VARIETY": "Habanero", "SPECIES": "C. chinense", "SHU": "100,000-350,000", "HS_Fresh": "0709.60.99", "HS_Dry": "0904.21.90", "HS_Ground": "0904.22.00", "Cuisine": "Caribbean", "Trivia": "Thin skin; very fruity aroma"},
    {"VARIETY": "Scotch Bonnet", "SPECIES": "C. chinense", "SHU": "100,000-350,000", "HS_Fresh": "0709.60.99", "HS_Dry": "0904.21.90", "HS_Ground": "0904.22.00", "Cuisine": "Caribbean", "Trivia": "Essential for Jerk spice"},
    {"VARIETY": "Bhut Jolokia (Ghost)", "SPECIES": "C. chinense", "SHU": "1,000,000+", "HS_Fresh": "0709.60.99", "HS_Dry": "0904.21.90", "HS_Ground": "0904.22.00", "Cuisine": "Indian", "Trivia": "First record breaker over 1M SHU"},
    {"VARIETY": "Aji Amarillo", "SPECIES": "C. baccatum", "SHU": "30,000-50,000", "HS_Fresh": "0709.60.99", "HS_Dry": "0904.21.90", "HS_Ground": "0904.22.00", "Cuisine": "Peruvian", "Trivia": "Bright orange; tastes like sunshine"},
    {"VARIETY": "Tabasco", "SPECIES": "C. frutescens", "SHU": "30,000-50,000", "HS_Fresh": "0709.60.99", "HS_Sauce": "2103.90.90", "HS_Ground": "N/A", "Cuisine": "American/Cajun", "Trivia": "Juiciest of the main species"},
    {"VARIETY": "Rocoto", "SPECIES": "C. pubescens", "SHU": "30,000-100,000", "HS_Fresh": "0709.60.99", "HS_Dry": "N/A", "HS_Ground": "N/A", "Cuisine": "Andean", "Trivia": "Black seeds and hairy leaves"},
    {"VARIETY": "Peperoncino", "SPECIES": "C. annuum", "SHU": "15,000-30,000", "HS_Fresh": "0709.60.99", "HS_Dry": "0904.21.90", "HS_Ground": "0904.22.00", "Cuisine": "Italian", "Trivia": "Calabrian spice base"},
    {"VARIETY": "Baklouti", "SPECIES": "C. annuum", "SHU": "1,000-5,000", "HS_Fresh": "0709.60.99", "HS_Sauce": "2103.90.90", "HS_Ground": "N/A", "Cuisine": "Maghreb", "Trivia": "Used for authentic Harissa paste"},
    {"VARIETY": "Piri-piri", "SPECIES": "C. frutescens", "SHU": "50,000-175,000", "HS_Fresh": "0709.60.99", "HS_Sauce": "2103.90.90", "HS_Ground": "0904.22.00", "Cuisine": "African/Portuguese", "Trivia": "African Bird's Eye"},
    {"VARIETY": "Padrón", "SPECIES": "C. annuum", "SHU": "500-2,500", "HS_Fresh": "0709.60.10", "HS_Preserve": "2001.90.97", "HS_Ground": "N/A", "Cuisine": "Spanish", "Trivia": "Some are hot, most are mild"},
    {"VARIETY": "Anaheim", "SPECIES": "C. annuum", "SHU": "500-1,000", "HS_Fresh": "0709.60.10", "HS_Dry": "0904.21.40", "HS_Ground": "0904.22.40", "Cuisine": "Californian/Mexican", "Trivia": "Mild and large; used for stuffing"},
    {"VARIETY": "Serrano", "SPECIES": "C. annuum", "SHU": "10,000-23,000", "HS_Fresh": "0709.60.91", "HS_Dry": "0904.21.90", "HS_Ground": "N/A", "Cuisine": "Mexican", "Trivia": "Sharper heat than Jalapeño"},
    {"VARIETY": "Guajillo", "SPECIES": "C. annuum", "SHU": "2,500-5,000", "HS_Fresh": "N/A", "HS_Dry": "0904.21.90", "HS_Ground": "0904.22.00", "Cuisine": "Mexican", "Trivia": "Smooth, dark skin; tea-like flavor"},
    {"VARIETY": "Bird's Eye (Thai)", "SPECIES": "C. annuum", "SHU": "50,000-100,000", "HS_Fresh": "0709.60.99", "HS_Dry": "0904.21.90", "HS_Ground": "0904.22.00", "Cuisine": "Thai/SE Asian", "Trivia": "Small but very punchy"},
    {"VARIETY": "Carolina Reaper", "SPECIES": "C. chinense", "SHU": "1,500,000-2,200,000", "HS_Fresh": "0709.60.99", "HS_Sauce": "2103.90.90", "HS_Ground": "0904.22.00", "Cuisine": "Novelty", "Trivia": "Current hottest in world (Ed Currie)"},
    {"VARIETY": "Pasilla", "SPECIES": "C. annuum", "SHU": "1,000-2,500", "HS_Fresh": "N/A", "HS_Dry": "0904.21.40", "HS_Ground": "0904.22.40", "Cuisine": "Mexican", "Trivia": "A.K.A. 'Chilaca' when fresh"}
]


# Make  Variety df
df = pd.DataFrame(pepper_data)

# Print all 23 entries
print(f"Total entries: {len(df)}")
print(df.to_string())
varietites_df = pd.DataFrame(pepper_data)

# Print result
print(varietites_df)


Total entries: 23
                 VARIETY        SPECIES                  SHU    HS_Fresh      HS_Dry   HS_Ground              Cuisine                                       Trivia    HS_Sauce HS_Preserve
0            Bell pepper      C. annuum                    0  0709.60.10  0904.21.60  0904.22.20               Global                      No heat; high vitamin C         NaN         NaN
1        Paprika (Sweet)      C. annuum              0-2,500  0709.60.10  0904.21.20  0904.22.20            Hungarian             Specific cultivar: 'Aji Paprika'         NaN         NaN
2                Cayenne      C. annuum        30,000-50,000  0709.60.99  0904.21.90  0904.22.00               Global             Standard for 'red pepper flakes'         NaN         NaN
3               Jalapeño      C. annuum          2,500-8,000  0709.60.91  0904.21.60         N/A               Mexico           Often pickled or smoked (Chipotle)         NaN         NaN
4          Poblano/Ancho      C. annuum        

# Popular Varieties by Heat Level

Mild (0-5,000 SHU): Bell Pepper, Pepperoncini, Poblano, Anaheim
Medium (5,000-30,000 SHU): Jalapeno, Serrano, Cascabel
Hot (30,000-100,000 SHU): Cayenne, Bird's Eye (Thai), Piri Piri
Very Hot/Superhot (>100,000 SHU): Habanero, Scotch Bonnet, Bhut Jolokia (Ghost), Trinidad Scorpion, Carolina Reaper

# Note: The SCOVILLE HEAT UNIT scale (SHU) is a LOG scale! 
# Take a moment then to imagine what a 1-2 MILLION Scoville pepper is like compared to milder Jalapeno! 
# Used an avg. value of estimated SHU for each Variety

# Codify "Pungency Index" of a country: Average SHU of peppers commonly used in a cuisine (e.g., Mexican, Turkish, Thai)

# Flavor Profiles: Fruity, Smoky, Citrus, Floral, Herbal, Sweet...
# Heat Profiles: Sharp, Pungent, Lingering, Burning, Triggering pain via trigeminal receptors...

# CHILI TYPES LEXICON - General Infos and Links 

--> There are several online Chili community websites, with quite botanical infos on Types/Varieties, maybe we can grab some of these sources?

Online chili lexicons categorize the over 3,000–4,000 varieties of peppers (genus Capsicum) into 
five primary domesticated species, though many others exist. 
These varieties are classified by heat level, shape, flavor, or species, with Capsicum annuum being the most common, including jalapeños and cayenne.

Example Resources: 
https://plantura.garden/uk/vegetables/chillies/types-of-chillies 
https://chili-plant.com/chilli-varieties/
https://chili-plant.com/interesting-facts/
https://pepperworld.com/harissa-selber-machen-herkunft-verwendung-rezepte/

https://wits.worldbank.org/trade/comtrade/en/country/ALL/year/2024/tradeflow/Imports/partner/WLD/product/070960 
https://www.freightamigo.com/en/blog/hs-code/hs-code-for-chili-peppers/
https://search.gesis.org/research_data/SDN-10.7802-2394


# Novelty Varieties prominent in Online Lexicons:

Aji Amarillo: Sunny yellow chili popular in Peruvian cuisine.
Black Pearl: Decorative pepper with black-to-red berries.
Bulgarian Carrot: Bright orange, thick-fleshed, carrot-shaped pepper.
Corno di Toro: "Bull’s horn" shaped pepper, often used for roasting.
Fish Pepper: Stripped green/white pepper, often used in fish dishes.
Peter Pepper: Unique for its distinctive, often described as "phallic" shape. 
